In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

EmbeddingVector = object
PAIRWISE_EMBEDDINGS_PD = pd.DataFrame({"embedding": [np.array([0.1, 0.2, 0.3]), np.array([0.4, 0.5, 0.6])]}, index=[101, 102])
PAIRWISE_TITLES_PD = pd.DataFrame({"title": [["paper_A", "paper_B"], ["paper_C"]]}, index=[101, 102])
PAIRWISE_TITLES_PL = pl.DataFrame({"document_id": [101, 102], "title": [["paper_A", "paper_B"], ["paper_C"]]})
PAIRWISE_EMBEDDINGS_PL = pl.DataFrame({"document_id": [101, 102], "embedding": [np.array([0.1, 0.2, 0.3]), np.array([0.4, 0.5, 0.6])]})
df = PAIRWISE_TITLES_PD

# --- pairwise_cosine_loc ---
FIX_PAIRWISE_COSINE_LOC_COL_EMBEDDING = np.array([0.4, 0.5, 0.6])
FIX_PAIRWISE_COSINE_LOC_DOCUMENT_ID_1 = 101
FIX_PAIRWISE_COSINE_LOC_DOCUMENT_ID_2 = 102
FIX_PAIRWISE_COSINE_LOC_ROW_EMBEDDING = np.array([0.1, 0.2, 0.3])

# --- pairwise_loc_filter ---
FIX_PAIRWISE_LOC_FILTER_COL_VALUE_LIST = ["paper_C"]
FIX_PAIRWISE_LOC_FILTER_COLNAME = "title"
FIX_PAIRWISE_LOC_FILTER_DOCUMENT_ID_1 = 101
FIX_PAIRWISE_LOC_FILTER_DOCUMENT_ID_2 = 102
FIX_PAIRWISE_LOC_FILTER_ROW_VALUE_LIST = ["paper_A", "paper_B"]

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_pairwise_cosine_loc(col_embedding, document_id_1, document_id_2, row_embedding):
    row_embedding: EmbeddingVector = df.loc[document_id_1].item()
    col_embedding: EmbeddingVector = df.loc[document_id_2].item()
    return None

def before_pairwise_loc_filter(col_value_list, colname, document_id_1, document_id_2, row_value_list):
    row_value_list: list[str] = df.loc[document_id_1, colname]
    col_value_list: list[str] = df.loc[document_id_2, colname]
    return None

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_pairwise_cosine_loc(col_embedding, document_id_1, document_id_2, row_embedding):

    _index_col = "index" if "index" in df.columns else df.columns[0]

    row_embedding: EmbeddingVector = (
        df.filter(pl.col(_index_col) == document_id_1)
        .drop(_index_col)
        .to_series(0)
        .item()
    )
    col_embedding: EmbeddingVector = (
        df.filter(pl.col(_index_col) == document_id_2)
        .drop(_index_col)
        .to_series(0)
        .item()
    )
    return None

def gen_pairwise_loc_filter(col_value_list, colname, document_id_1, document_id_2, row_value_list):

    row_value_list: list[str] = df.filter(pl.col("index") == document_id_1).get_column(colname).to_list()
    col_value_list: list[str] = df.filter(pl.col("index") == document_id_2).get_column(colname).to_list()
    return None

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: pairwise_loc_filter ===

def _capture_return_locals(_fn, *args):
    import sys
    _captured = {}
    _code = _fn.__code__
    def _trace(_frame, _event, _arg):
        if _event == "return" and _frame.f_code is _code:
            _captured.update(_frame.f_locals)
        return _trace
    _previous = sys.gettrace()
    try:
        sys.settrace(_trace)
        _fn(*args)
    finally:
        sys.settrace(_previous)
    return _captured

try:
    df = PAIRWISE_TITLES_PL
    _r = gen_pairwise_loc_filter(
        FIX_PAIRWISE_LOC_FILTER_COL_VALUE_LIST, "title", 101, 102,
        FIX_PAIRWISE_LOC_FILTER_ROW_VALUE_LIST,
    )
    print("✅ L1 smoke gen_pairwise_loc_filter: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_pairwise_loc_filter: {type(_e).__name__}: {_e}")

try:
    df = PAIRWISE_TITLES_PD
    _rb = before_pairwise_loc_filter(
        FIX_PAIRWISE_LOC_FILTER_COL_VALUE_LIST, "title", 101, 102,
        FIX_PAIRWISE_LOC_FILTER_ROW_VALUE_LIST,
    )
    print("✅ L1 smoke before_pairwise_loc_filter: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_pairwise_loc_filter: {type(_e).__name__}: {_e}")

# L2 compares the selected list values captured before function return.
try:
    df = PAIRWISE_TITLES_PD
    _before_locals = _capture_return_locals(
        before_pairwise_loc_filter,
        FIX_PAIRWISE_LOC_FILTER_COL_VALUE_LIST, "title", 101, 102,
        FIX_PAIRWISE_LOC_FILTER_ROW_VALUE_LIST,
    )
    df = PAIRWISE_TITLES_PL
    _gen_locals = _capture_return_locals(
        gen_pairwise_loc_filter,
        FIX_PAIRWISE_LOC_FILTER_COL_VALUE_LIST, "title", 101, 102,
        FIX_PAIRWISE_LOC_FILTER_ROW_VALUE_LIST,
    )
    _before_values = (
        _before_locals["row_value_list"], _before_locals["col_value_list"]
    )
    _gen_values = (
        _gen_locals["row_value_list"], _gen_locals["col_value_list"]
    )
    if _before_values == _gen_values:
        print("✅ L2 equivalence pairwise_loc_filter: MATCH")
    else:
        print(
            "❌ L2 equivalence pairwise_loc_filter: MISMATCH - "
            f"before={_before_values}, gen={_gen_values}"
        )
except Exception as _e:
    print(f"❌ L2 equivalence pairwise_loc_filter: setup error - {type(_e).__name__}: {_e}")

# L3 compares semantic missing-id behavior.
try:
    _before_err = _gen_err = None
    try:
        df = PAIRWISE_TITLES_PD
        before_pairwise_loc_filter([], "title", 999, 102, [])
    except Exception as _e:
        _before_err = _e
    try:
        df = PAIRWISE_TITLES_PL
        gen_pairwise_loc_filter([], "title", 999, 102, [])
    except Exception as _e:
        _gen_err = _e
    if _before_err is not None and _gen_err is not None and not isinstance(_gen_err, (SyntaxError, NameError)):
        print("✅ L3 edge pairwise_loc_filter missing id: MATCH - both rejected")
    elif _before_err is None and _gen_err is None:
        print("✅ L3 edge pairwise_loc_filter missing id: MATCH - both accepted")
    else:
        print(
            "❌ L3 edge pairwise_loc_filter missing id: MISMATCH - "
            f"before_error={type(_before_err).__name__ if _before_err else None}, "
            f"gen_error={type(_gen_err).__name__ if _gen_err else None}"
        )
except Exception as _e:
    print(f"❌ L3 edge pairwise_loc_filter missing id: setup error - {type(_e).__name__}: {_e}")


❌ L1 smoke gen_pairwise_loc_filter: ColumnNotFoundError: unable to find column "index"; valid columns: ["document_id", "title"]
✅ L1 smoke before_pairwise_loc_filter: OK
❌ L2 equivalence pairwise_loc_filter: setup error - ColumnNotFoundError: unable to find column "index"; valid columns: ["document_id", "title"]
✅ L3 edge pairwise_loc_filter missing id: MATCH - both rejected
